# 02 — Feature Extraction & Classification

Phase 1 of the research: extract SAE feature activations from Gemma 2 9B IT
processing MGSM math problems across 5 languages, then classify features into
language-specific, reasoning-specific, and shared categories.

**Methods:**
1. Monolinguality metric (Deng et al. 2025)
2. Supervised language probe (logistic regression)
3. Cross-lingual stability (reasoning features)

**Run on Colab with A100 GPU.** Estimated runtime: ~45 min for full extraction.

## 0. Install & Setup

In [ ]:
# Run this cell on Colab, then RESTART RUNTIME before continuing
# (Runtime → Restart session)
!pip install -q transformers sae-lens nnsight python-dotenv tqdm scikit-learn sentence-transformers

In [ ]:
import sys
import os

# Set HF token from Colab secrets
from google.colab import userdata
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
print(f"HF_TOKEN set: {'yes' if os.environ.get('HF_TOKEN') else 'NO — add it to Colab secrets!'}")

# Clone repo or ensure src/ is available
if not os.path.exists('src'):
    print('src/ not found — clone the repo or upload src/ directory')
else:
    # Install in dev mode so 'from src.X import Y' works
    !pip install -q -e .
    print('Installed project in dev mode')

if '.' not in sys.path:
    sys.path.insert(0, '.')

In [ ]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected. This notebook requires an A100.")

In [ ]:
from pathlib import Path

RESULTS_DIR = Path('results')
RESULTS_DIR.mkdir(exist_ok=True)
print(f"Results will be saved to: {RESULTS_DIR.resolve()}")

## 1. Load Model & Data

Model: **Gemma 2 9B IT** (verified in notebook 01).
We load the model directly here rather than importing from `src/model.py`
(which has stale Gemma 3 4B references).

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

# ── Constants (verified in 01_setup_verify_fixed) ──
MODEL_ID = 'google/gemma-2-9b-it'  # NOT gemma-3-4b-it
TARGET_LANGUAGES = ['en', 'zh', 'es', 'bn', 'sw']

# SAE constants — from Colab-verified HF repo listing
SAE_RELEASE = 'gemma-scope-9b-it-res-canonical'
SAE_LAYERS = [9, 20, 31]  # NOT [9, 17, 22, 29]
SAE_WIDTH_STR = '16k'     # also available: '131k'
SAE_D_SAE = 16384         # feature count for 16k width

token = os.environ.get('HF_TOKEN')
assert token, "Set HF_TOKEN in Colab secrets first!"

print(f"Loading {MODEL_ID}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=token)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map='auto',
    token=token,
)
model.eval()

print(f"Model loaded. Parameters: {sum(p.numel() for p in model.parameters()) / 1e9:.2f}B")
print(f"Hidden size: {model.config.hidden_size}")
print(f"Layers: {len(model.model.layers)}")

In [ ]:
from src.data import load_mgsm  # model-agnostic, still works

print("Loading MGSM dataset...")
mgsm_data = load_mgsm(TARGET_LANGUAGES)

for lang in TARGET_LANGUAGES:
    print(f"  {lang}: {len(mgsm_data[lang])} examples")

# Verify same problems across languages (gold answers should match)
for i in range(3):
    answers = [mgsm_data[lang][i]['answer_number'] for lang in TARGET_LANGUAGES]
    assert len(set(answers)) == 1, f"Problem {i} answers differ: {answers}"
print(f"\nVerified: first 3 problems have matching gold answers across languages.")

In [ ]:
# Format all prompts using tokenizer.apply_chat_template
prompts_by_lang = {}

for lang in TARGET_LANGUAGES:
    prompts = []
    for ex in mgsm_data[lang]:
        messages = [{'role': 'user', 'content': ex['question']}]
        prompt = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        prompts.append(prompt)
    prompts_by_lang[lang] = prompts

total = sum(len(v) for v in prompts_by_lang.values())
print(f"Formatted {total} prompts ({len(TARGET_LANGUAGES)} langs x {len(prompts_by_lang['en'])} each)")

# Show an example
print(f"\nExample prompt (en, problem 0):")
print(prompts_by_lang['en'][0][:200] + '...')

## 2. Extract Activations & Encode Through SAEs

Process **one layer at a time** to manage GPU memory:
1. Load SAE for this layer (16k width, canonical) via SAELens
2. For each language: extract last-token residual activations via nnsight, encode through SAE
3. Save features to disk, free SAE

**Key API notes** (from 01_setup_verify_fixed):
- nnsight trace: must unpack inputs with `**` — `nn_model.trace(**inputs, ...)`
- nnsight output shape: `(seq_len, d_model)` — no batch dim, no `.value` accessor
- SAE loading: release = `gemma-scope-9b-it-res-canonical`, sae_id uses slashes
- Must cast activations to SAE's dtype before encoding

In [ ]:
from sae_lens import SAE
from nnsight import NNsight
from tqdm import tqdm
import gc
import time

LAYERS = SAE_LAYERS  # [9, 20, 31]

print(f"Extraction plan:")
print(f"  Layers: {LAYERS}")
print(f"  SAE release: {SAE_RELEASE}")
print(f"  SAE width: {SAE_WIDTH_STR} ({SAE_D_SAE} features)")
print(f"  Languages: {TARGET_LANGUAGES}")
print(f"  Examples per language: {len(prompts_by_lang['en'])}")

In [ ]:
def extract_last_token_activations(model, tokenizer, texts, layer, device='cuda'):
    """Extract last-token residual stream activations at a single layer.

    Uses nnsight >= 0.3 API:
    - Must unpack tokenizer output with ** into trace()
    - Saved proxy IS the tensor (no .value accessor)
    - Output shape is (seq_len, d_model) for single inputs (no batch dim)
    """
    nn_model = NNsight(model)
    all_acts = []

    for text in tqdm(texts, desc=f"Layer {layer}"):
        inputs = tokenizer(text, return_tensors='pt').to(device)

        with nn_model.trace(**inputs, scan=False, validate=False):
            resid = nn_model.model.layers[layer].output[0].save()

        # resid shape: (seq_len, d_model) — no batch dim, no .value
        last_token = resid[-1].cpu()  # (d_model,)
        all_acts.append(last_token)

    return torch.stack(all_acts)  # (n_texts, d_model)


def encode_through_sae(activations, sae, batch_size=64):
    """Encode activations through SAE, handling dtype/device automatically."""
    sae_device = next(sae.parameters()).device
    sae_dtype = next(sae.parameters()).dtype

    all_features = []
    for i in range(0, len(activations), batch_size):
        batch = activations[i:i + batch_size].to(device=sae_device, dtype=sae_dtype)
        with torch.no_grad():
            features = sae.encode(batch)
        all_features.append(features.cpu())

    return torch.cat(all_features)  # (n_texts, d_sae)


print("Extraction functions defined.")

In [ ]:
total_start = time.time()

for layer in LAYERS:
    layer_start = time.time()
    print(f"\n{'='*60}")
    print(f"LAYER {layer}")
    print(f"{'='*60}")

    # Check if already extracted (for resuming)
    all_exist = all(
        (RESULTS_DIR / f'features_{lang}_layer{layer:02d}_{SAE_WIDTH_STR}.pt').exists()
        for lang in TARGET_LANGUAGES
    )
    if all_exist:
        print(f"  All feature files for layer {layer} already exist, skipping.")
        continue

    # Step 1: Load SAE for this layer
    sae_id = f'layer_{layer}/width_{SAE_WIDTH_STR}/canonical'
    print(f"  Loading SAE: {SAE_RELEASE} / {sae_id}")
    sae = SAE.from_pretrained(release=SAE_RELEASE, sae_id=sae_id, device='cuda')
    print(f"  SAE loaded: d_in={sae.cfg.d_in}, d_sae={sae.cfg.d_sae}")

    # Step 2: Extract activations and encode for each language
    for lang in TARGET_LANGUAGES:
        save_path = RESULTS_DIR / f'features_{lang}_layer{layer:02d}_{SAE_WIDTH_STR}.pt'
        if save_path.exists():
            print(f"  {lang}: already exists, skipping.")
            continue

        print(f"  {lang}: extracting residual activations...")
        acts = extract_last_token_activations(
            model, tokenizer, prompts_by_lang[lang], layer=layer,
        )
        print(f"    Residual activations: {acts.shape}")  # (250, 3584)

        print(f"  {lang}: encoding through SAE...")
        features = encode_through_sae(acts, sae)
        print(f"    SAE features: {features.shape}")  # (250, 16384)
        n_active = (features > 0).float().mean(dim=0).gt(0).sum().item()
        print(f"    Features active in >=1 example: {n_active}/{features.shape[1]}")

        # Save
        torch.save(features, save_path)
        print(f"    Saved to {save_path}")

        # Free activation memory
        del acts, features
        torch.cuda.empty_cache()

    # Step 3: Free SAE before next layer
    del sae
    gc.collect()
    torch.cuda.empty_cache()

    elapsed = time.time() - layer_start
    print(f"  Layer {layer} done in {elapsed:.0f}s")

total_elapsed = time.time() - total_start
print(f"\nAll extraction complete in {total_elapsed/60:.1f} min")

In [ ]:
# Verify saved files
print("Saved feature files:")
for layer in LAYERS:
    for lang in TARGET_LANGUAGES:
        path = RESULTS_DIR / f'features_{lang}_layer{layer:02d}_{SAE_WIDTH_STR}.pt'
        if path.exists():
            t = torch.load(path, weights_only=True)
            print(f"  {path.name}: {t.shape}")
        else:
            print(f"  {path.name}: MISSING")

## 3. Feature Classification

For each layer, classify features using three methods:
- **3a.** Monolinguality metric
- **3b.** Supervised language probe
- **3c.** Cross-lingual reasoning features

These functions from `src/monolinguality.py` are model-agnostic (operate on feature tensors only).

In [ ]:
from src.monolinguality import (
    compute_monolinguality,
    identify_language_features,
    identify_reasoning_features,
    train_language_probe,
    probe_language_features,
)

### 3a. Monolinguality Metric

In [ ]:
TOP_K = 50

mono_results = {}  # layer -> {monolinguality scores, top features}

for layer in LAYERS:
    print(f"\n--- Layer {layer}: Monolinguality ---")

    # Load features for all languages at this layer
    features_by_lang = {}
    for lang in TARGET_LANGUAGES:
        path = RESULTS_DIR / f'features_{lang}_layer{layer:02d}_{SAE_WIDTH_STR}.pt'
        features_by_lang[lang] = torch.load(path, weights_only=True)

    # Compute monolinguality scores
    mono_scores = compute_monolinguality(features_by_lang)

    # Identify top language-specific features
    lang_features = identify_language_features(mono_scores, top_k=TOP_K)

    mono_results[layer] = {
        'scores': mono_scores,
        'top_features': lang_features,
    }

    # Print summary
    for lang in TARGET_LANGUAGES:
        top_score = mono_scores[lang][lang_features[lang][0]].item()
        print(f"  {lang}: top feature #{lang_features[lang][0]} (score={top_score:.4f})")

    # Save
    save_data = {
        'scores': {lang: s.cpu() for lang, s in mono_scores.items()},
        'top_features': lang_features,
    }
    torch.save(save_data, RESULTS_DIR / f'monolinguality_layer{layer:02d}.pt')

print("\nMonolinguality results saved.")

### 3b. Supervised Language Probe

In [ ]:
import numpy as np

probe_results = {}  # layer -> {clf, importances, top features, accuracy}

for layer in LAYERS:
    print(f"\n--- Layer {layer}: Language Probe ---")

    # Load features
    features_by_lang = {}
    for lang in TARGET_LANGUAGES:
        path = RESULTS_DIR / f'features_{lang}_layer{layer:02d}_{SAE_WIDTH_STR}.pt'
        features_by_lang[lang] = torch.load(path, weights_only=True)

    # Train probe
    clf, importances = train_language_probe(features_by_lang)
    languages_sorted = sorted(TARGET_LANGUAGES)

    # Probe accuracy (on training data — just to verify it works)
    X_all = np.concatenate([features_by_lang[l].float().numpy() for l in languages_sorted])
    y_all = np.concatenate([np.full(len(features_by_lang[l]), i) for i, l in enumerate(languages_sorted)])
    train_acc = clf.score(X_all, y_all)
    print(f"  Probe training accuracy: {train_acc:.1%} (chance = {1/len(TARGET_LANGUAGES):.1%})")

    # Extract top features per language
    probe_features = probe_language_features(clf, importances, languages_sorted, top_k=TOP_K)

    probe_results[layer] = {
        'clf': clf,
        'importances': importances,
        'top_features': probe_features,
        'accuracy': train_acc,
        'languages': languages_sorted,
    }

    for lang in languages_sorted:
        print(f"  {lang}: top probe feature #{probe_features[lang][0]}")

    # Save (excluding sklearn object — save importances and features)
    save_data = {
        'importances': importances,
        'top_features': probe_features,
        'accuracy': train_acc,
        'languages': languages_sorted,
    }
    torch.save(save_data, RESULTS_DIR / f'probe_layer{layer:02d}.pt')

print("\nProbe results saved.")

### 3c. Reasoning Features (Cross-Lingual Stability)

In [ ]:
reasoning_results = {}  # layer -> list of feature indices

for layer in LAYERS:
    print(f"\n--- Layer {layer}: Reasoning Features ---")

    # Load features
    features_by_lang = {}
    for lang in TARGET_LANGUAGES:
        path = RESULTS_DIR / f'features_{lang}_layer{layer:02d}_{SAE_WIDTH_STR}.pt'
        features_by_lang[lang] = torch.load(path, weights_only=True)

    # Identify cross-lingual reasoning features
    reasoning_feats = identify_reasoning_features(features_by_lang)
    reasoning_results[layer] = reasoning_feats

    print(f"  Reasoning features found: {len(reasoning_feats)}")
    if reasoning_feats:
        print(f"  First 10: {reasoning_feats[:10]}")

    # Save
    torch.save(reasoning_feats, RESULTS_DIR / f'reasoning_features_layer{layer:02d}.pt')

print("\nReasoning feature results saved.")

## 4. Analysis & Visualization

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

### 4a. Feature Taxonomy Per Layer

In [ ]:
# Build a feature taxonomy for each layer:
# - language-specific (union of top-K across all langs from monolinguality)
# - reasoning-specific
# - shared (in both language and reasoning sets)
# - unclassified (neither)

taxonomy = {}

for layer in LAYERS:
    # Language features: union across all languages
    lang_feat_set = set()
    for lang in TARGET_LANGUAGES:
        lang_feat_set.update(mono_results[layer]['top_features'][lang])

    reasoning_feat_set = set(reasoning_results[layer])

    shared = lang_feat_set & reasoning_feat_set
    lang_only = lang_feat_set - shared
    reasoning_only = reasoning_feat_set - shared

    taxonomy[layer] = {
        'language_only': len(lang_only),
        'reasoning_only': len(reasoning_only),
        'shared': len(shared),
        'total_language': len(lang_feat_set),
        'total_reasoning': len(reasoning_feat_set),
    }

    print(f"Layer {layer}:")
    print(f"  Language-specific features: {len(lang_feat_set)} (union of top-{TOP_K} per lang)")
    print(f"  Reasoning features: {len(reasoning_feat_set)}")
    print(f"  Shared (both): {len(shared)}")
    print(f"  Language-only: {len(lang_only)}")
    print(f"  Reasoning-only: {len(reasoning_only)}")
    print()

In [ ]:
# Bar chart: feature counts per layer
fig, ax = plt.subplots(figsize=(8, 5))

x = range(len(LAYERS))
bar_width = 0.25

lang_counts = [taxonomy[l]['language_only'] for l in LAYERS]
reasoning_counts = [taxonomy[l]['reasoning_only'] for l in LAYERS]
shared_counts = [taxonomy[l]['shared'] for l in LAYERS]

ax.bar([i - bar_width for i in x], lang_counts, bar_width, label='Language-only', color='#e74c3c')
ax.bar(x, reasoning_counts, bar_width, label='Reasoning-only', color='#3498db')
ax.bar([i + bar_width for i in x], shared_counts, bar_width, label='Shared', color='#2ecc71')

ax.set_xlabel('Layer')
ax.set_ylabel('Number of Features')
ax.set_title('Feature Taxonomy by Layer')
ax.set_xticks(x)
ax.set_xticklabels([str(l) for l in LAYERS])
ax.legend()
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'feature_taxonomy_by_layer.png', bbox_inches='tight')
plt.show()

### 4b. Method Agreement (Monolinguality vs Probe)

In [ ]:
# Jaccard overlap between monolinguality and probe top-K features
print("Jaccard overlap between monolinguality and probe (top-50 per lang):")
print(f"{'Layer':<8} {'Lang':<6} {'Jaccard':<10} {'Overlap':<10}")
print('-' * 34)

agreement_data = {}

for layer in LAYERS:
    agreement_data[layer] = {}
    for lang in TARGET_LANGUAGES:
        mono_set = set(mono_results[layer]['top_features'][lang])
        probe_set = set(probe_results[layer]['top_features'].get(lang, []))

        if mono_set or probe_set:
            intersection = mono_set & probe_set
            union = mono_set | probe_set
            jaccard = len(intersection) / len(union) if union else 0.0
        else:
            jaccard = 0.0
            intersection = set()

        agreement_data[layer][lang] = jaccard
        print(f"{layer:<8} {lang:<6} {jaccard:<10.3f} {len(intersection):<10}")

In [ ]:
# Heatmap of Jaccard overlap
fig, ax = plt.subplots(figsize=(7, 4))

data_matrix = []
for layer in LAYERS:
    row = [agreement_data[layer][lang] for lang in TARGET_LANGUAGES]
    data_matrix.append(row)

im = ax.imshow(data_matrix, cmap='YlOrRd', aspect='auto', vmin=0, vmax=1)
ax.set_xticks(range(len(TARGET_LANGUAGES)))
ax.set_xticklabels(TARGET_LANGUAGES)
ax.set_yticks(range(len(LAYERS)))
ax.set_yticklabels([str(l) for l in LAYERS])
ax.set_xlabel('Language')
ax.set_ylabel('Layer')
ax.set_title('Mono vs Probe Agreement (Jaccard)')

# Annotate cells
for i in range(len(LAYERS)):
    for j in range(len(TARGET_LANGUAGES)):
        ax.text(j, i, f'{data_matrix[i][j]:.2f}', ha='center', va='center', fontsize=9)

plt.colorbar(im, ax=ax, label='Jaccard')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'method_agreement_heatmap.png', bbox_inches='tight')
plt.show()

### 4c. Per-Language Breakdown

In [ ]:
# Average monolinguality score of top-K features per language per layer
print("Average monolinguality score of top-50 features:")
print(f"{'Layer':<8}", ''.join(f'{lang:<10}' for lang in TARGET_LANGUAGES))
print('-' * 58)

lang_strength = {lang: [] for lang in TARGET_LANGUAGES}

for layer in LAYERS:
    row = f"{layer:<8}"
    for lang in TARGET_LANGUAGES:
        scores = mono_results[layer]['scores'][lang]
        top_feats = mono_results[layer]['top_features'][lang]
        avg_score = scores[top_feats].mean().item()
        lang_strength[lang].append(avg_score)
        row += f"{avg_score:<10.4f}"
    print(row)

In [ ]:
# Line plot: monolinguality strength across layers
fig, ax = plt.subplots(figsize=(8, 5))

for lang in TARGET_LANGUAGES:
    ax.plot(LAYERS, lang_strength[lang], marker='o', label=lang)

ax.set_xlabel('Layer')
ax.set_ylabel('Avg Monolinguality Score (top-50)')
ax.set_title('Language Feature Strength Across Layers')
ax.set_xticks(LAYERS)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'monolinguality_across_layers.png', bbox_inches='tight')
plt.show()

### 4d. Layer-Wise Trends

In [ ]:
# Probe accuracy and reasoning feature count across layers
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Probe accuracy
probe_accs = [probe_results[l]['accuracy'] for l in LAYERS]
ax1.bar([str(l) for l in LAYERS], probe_accs, color='#9b59b6')
ax1.axhline(y=1/len(TARGET_LANGUAGES), color='gray', linestyle='--', label=f'Chance ({1/len(TARGET_LANGUAGES):.0%})')
ax1.set_xlabel('Layer')
ax1.set_ylabel('Accuracy')
ax1.set_title('Language Probe Accuracy by Layer')
ax1.set_ylim(0, 1.05)
ax1.legend()

# Reasoning feature count
reasoning_counts_layer = [len(reasoning_results[l]) for l in LAYERS]
ax2.bar([str(l) for l in LAYERS], reasoning_counts_layer, color='#3498db')
ax2.set_xlabel('Layer')
ax2.set_ylabel('Count')
ax2.set_title('Reasoning Features by Layer')

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'layer_trends.png', bbox_inches='tight')
plt.show()

print("Probe accuracies:", {l: f"{a:.1%}" for l, a in zip(LAYERS, probe_accs)})
print("Reasoning feature counts:", {l: c for l, c in zip(LAYERS, reasoning_counts_layer)})

### 4e. Save Phase 1 Summary

In [ ]:
phase1_summary = {
    'model_id': MODEL_ID,
    'layers': LAYERS,
    'languages': TARGET_LANGUAGES,
    'top_k': TOP_K,
    'sae_release': SAE_RELEASE,
    'sae_width': SAE_WIDTH_STR,
    'sae_d_sae': SAE_D_SAE,
    'taxonomy': taxonomy,
    'agreement': agreement_data,
    'probe_accuracies': {l: probe_results[l]['accuracy'] for l in LAYERS},
    'reasoning_feature_counts': {l: len(reasoning_results[l]) for l in LAYERS},
    'mono_top_features': {l: mono_results[l]['top_features'] for l in LAYERS},
    'probe_top_features': {l: probe_results[l]['top_features'] for l in LAYERS},
    'reasoning_features': reasoning_results,
}

torch.save(phase1_summary, RESULTS_DIR / 'phase1_summary.pt')
print("Phase 1 summary saved to results/phase1_summary.pt")
print(f"\nKeys: {list(phase1_summary.keys())}")

## 5. Baseline Sanity Check

Run the model on 10 problems per language (no intervention) to establish baseline accuracy for later comparison.

`evaluate_mgsm()` from `src/evaluation.py` is model-agnostic (uses `model.generate()` directly).

In [ ]:
from src.evaluation import evaluate_mgsm  # model-agnostic, still works

N_EVAL = 10  # problems per language for quick baseline

baseline_results = {}

for lang in TARGET_LANGUAGES:
    print(f"\nEvaluating {lang} ({N_EVAL} problems)...")

    # Format prompts
    questions = prompts_by_lang[lang][:N_EVAL]
    gold_answers = [ex['answer_number'] for ex in mgsm_data[lang][:N_EVAL]]

    result = evaluate_mgsm(
        model, tokenizer, questions, gold_answers,
        max_new_tokens=512, batch_size=1,
    )
    baseline_results[lang] = result
    print(f"  Accuracy: {result['accuracy']:.0%} ({sum(result['correct'])}/{N_EVAL})")

print(f"\n{'='*40}")
print("Baseline Results (no intervention):")
print(f"{'='*40}")
for lang in TARGET_LANGUAGES:
    acc = baseline_results[lang]['accuracy']
    n = sum(baseline_results[lang]['correct'])
    print(f"  {lang}: {acc:.0%} ({n}/{N_EVAL})")

avg_acc = sum(r['accuracy'] for r in baseline_results.values()) / len(baseline_results)
print(f"  Average: {avg_acc:.0%}")

In [ ]:
# Save baseline results
baseline_save = {
    lang: {
        'accuracy': r['accuracy'],
        'predictions': r['predictions'],
        'correct': r['correct'],
    }
    for lang, r in baseline_results.items()
}
baseline_save['n_eval'] = N_EVAL
baseline_save['average_accuracy'] = avg_acc

torch.save(baseline_save, RESULTS_DIR / 'baseline_eval.pt')
print("Baseline results saved to results/baseline_eval.pt")

In [ ]:
# Final memory report
if torch.cuda.is_available():
    total = torch.cuda.get_device_properties(0).total_memory
    print(f"GPU Memory Allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
    print(f"GPU Memory Reserved:  {torch.cuda.memory_reserved() / 1e9:.2f} GB")
    print(f"GPU Memory Total:     {total / 1e9:.2f} GB")
    print(f"GPU Memory Free:      {(total - torch.cuda.memory_allocated()) / 1e9:.2f} GB")

## Summary

### Model & SAE
- Model: `google/gemma-2-9b-it` (d_model=3584, 42 layers)
- SAEs: `gemma-scope-9b-it-res-canonical`, width=16k, layers {9, 20, 31}

### Files produced
- `results/features_{lang}_layer{NN}_16k.pt` — SAE feature activations (250, 16384) per language per layer
- `results/monolinguality_layer{NN}.pt` — monolinguality scores and top-50 features
- `results/probe_layer{NN}.pt` — probe importances and top-50 features
- `results/reasoning_features_layer{NN}.pt` — reasoning feature indices
- `results/phase1_summary.pt` — aggregated taxonomy, agreement, and trends
- `results/baseline_eval.pt` — baseline accuracy (no intervention)

### Key questions answered
1. How many features are language-specific vs reasoning-specific at each layer?
2. Do the monolinguality metric and probe agree on which features are language-specific?
3. Which languages have the strongest language-specific representations?
4. How does feature composition change across layers (early → middle → late)?

### Next steps (Notebook 03)
- Use identified language features for targeted ablation experiments
- Compare SAE-guided ablation vs Zhao et al. SVD-based intervention
- Measure impact on accuracy, language fidelity, and perplexity